# Chapter 03. 데이터의 첫인상 읽기

수업용 가상 쇼핑몰 CSV 4개를 실제로 점검한 결과를 기록한 제출용 Notebook입니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
print('Python:', sys.executable)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)

Python: C:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
PROJECT_ROOT: C:\dev\llm-data-analysis-course
DATA_DIR: C:\dev\llm-data-analysis-course\data\raw


In [2]:
customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')
datasets = {'customers': customers, 'products': products, 'orders': orders, 'order_items': order_items}
for name, df in datasets.items():
    print(f'{name}: {df.shape}')

customers: (150, 6)
products: (100, 4)
orders: (300, 5)
order_items: (764, 5)


## STEP 1. 구조 확인

### 결과 관찰
customers는 150행 6열, products는 100행 4열, orders는 300행 5열, order_items는 764행 5열이다.

### 나의 해석과 판단
orders보다 order_items 행이 많은 것은 한 주문에 여러 상품 상세가 연결될 수 있기 때문으로 해석할 수 있다. 따라서 order_items의 order_id 반복을 단순 중복으로 삭제하면 안 된다.

In [3]:
for name, df in datasets.items():
    print(f'{name} missing values:', df.isna().sum().sum())
for name, key in {'customers':'customer_id', 'products':'product_id', 'orders':'order_id', 'order_items':'order_item_id'}.items():
    print(f'{key} duplicate:', datasets[name][key].duplicated().sum())

customers missing values: 0
products missing values: 0
orders missing values: 0
order_items missing values: 0
customer_id duplicate: 0
product_id duplicate: 0
order_id duplicate: 0
order_item_id duplicate: 0


## STEP 2. 결측·중복·ID 품질

### 결과 관찰
네 파일의 결측값 수는 모두 0이며, customers.customer_id, products.product_id, orders.order_id, order_items.order_item_id의 중복도 모두 0이다.

### 나의 해석과 판단
현재 샘플 데이터에서는 주요 식별자를 테이블 연결의 기준으로 사용할 수 있다. 그러나 ID의 고유성만으로 값 자체의 업무적 정확성까지 보장되지는 않는다.

In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
print('order_date range:', orders['order_date'].min().date(), 'to', orders['order_date'].max().date())
print('order_status:', orders['order_status'].value_counts().to_dict())
print('category counts:', products['category'].value_counts().to_dict())

order_date range: 2025-07-09 to 2026-07-08
order_status: cancelled 64, completed 184, refunded 52
category counts: 도서 14, 뷰티 16, 생활용품 16, 스포츠 19, 식품 7, 전자기기 17, 패션 11


## STEP 3. 숫자·범주·날짜 점검

### 결과 관찰
주문 날짜 범위는 2025-07-09부터 2026-07-08까지다. 주문 상태는 completed 184건, cancelled 64건, refunded 52건이다. 상품 카테고리는 7종이며 상품 수는 카테고리마다 다르다.

### 나의 해석과 판단
완료·취소·환불 주문을 함께 판매 금액으로 합치면 분석 목적과 다른 결과가 될 수 있다. 판매 분석에서는 먼저 주문 상태의 범위를 명시해야 한다.

In [5]:
print('orders.customer_id not in customers:', (~orders['customer_id'].isin(customers['customer_id'])).sum())
print('order_items.order_id not in orders:', (~order_items['order_id'].isin(orders['order_id'])).sum())
print('order_items.product_id not in products:', (~order_items['product_id'].isin(products['product_id'])).sum())

orders.customer_id not in customers: 0
order_items.order_id not in orders: 0
order_items.product_id not in products: 0


## STEP 4. CSV 간 키 관계

### 결과 관찰
orders의 customer_id, order_items의 order_id와 product_id에서 부모 테이블에 없는 값은 모두 0건이었다.

### 나의 해석과 판단
현재 확인 범위에서는 orphan key가 없으므로 네 CSV를 연결하는 기본 키 관계는 정상으로 보인다. 다만 이것은 업무 규칙과 값의 정확성 전체를 검증한 결과는 아니다.

## STEP 5. LLM 검증
LLM에는 파일명, 컬럼명, 행·열 수, 결측·중복 개수와 키 관계 확인 결과만 제공했다. 원본 고객 이름 등 개인정보가 될 수 있는 행 데이터는 제공하지 않았다. LLM의 제안은 실제 컬럼·계산 범위와 비교한 뒤 채택·수정·보류로 판단해야 한다.

## 최종 정리
1. 주요 ID의 결측과 중복은 현재 샘플 데이터에서 발견되지 않았다.
2. order_items는 주문 상세 테이블이므로 order_id 반복이 정상일 수 있다.
3. 이후 판매 분석에서는 completed 주문만 선택하고, 날짜 변환·merge 결과·집계 총합을 계속 검증해야 한다.

> Evidence 이미지는 실제 실행 화면을 캡처한 후 `data_analysis_assignments/chapter03/images/`에 추가한다.